# Job Offer Acceptance Prediction: Exploratory Data Analysis (EDA)

This notebook performs exploratory data analysis, data cleaning, domain feature engineering, and statistical feature selection on the Job Offer Acceptance dataset.

In [ ]:
# 1. Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier

import warnings
warnings.filterwarnings('ignore')
plt.style.use('ggplot')
print('Libraries imported successfully!')

In [ ]:
# 2. Load the Dataset
PATH = '../dataset/job_offer_acceptance_dataset.csv'
df = pd.read_csv(PATH)
df.head()

In [ ]:
# 3. Dataset Info
print(f'Dataset Shape: {df.shape}')
df.info()

In [ ]:
# 4. Check Missing Values & Duplicates
print('Missing Values:\n', df.isnull().sum())
print('\nDuplicate Rows:', df.duplicated().sum())

In [ ]:
# 5. Drop Unwanted / Noisy Features
# Dropping features that introduce noise or leak non-predictive metadata
features_to_drop = [
    'candidate_id', 'major', 'university_tier', 'city', 'education_level',
    'kaggle_score', 'linkedin_connections', 'internship_count',
    'github_repositories', 'hackathons_participated', 'leadership_experience',
    'company_industry', 'publications_count', 'application_month', 'application_year'
]

df = df.drop(columns=[c for c in features_to_drop if c in df.columns])
print('Cleaned Dataset Shape:', df.shape)
print('Remaining Active Features:', df.columns.tolist())
df.head()

In [ ]:
# 6. Target Variable Analysis
TARGET = 'offer_accepted'

print(df[TARGET].value_counts())
print('\nPercentage Distribution:')
print(df[TARGET].value_counts(normalize=True) * 100)

plt.figure(figsize=(7, 5))
sns.countplot(data=df, x=TARGET, palette=['#e74c3c', '#2ecc71'])
plt.title('Job Offer Acceptance Distribution')
plt.xlabel('Offer Accepted (0 = Rejected, 1 = Accepted)')
plt.ylabel('Number of Candidates')
plt.show()

In [ ]:
# 7. Summary Statistics of Numerical Columns
df.describe()

In [ ]:
# 8. Salary Distribution by Target Outcome
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
sns.histplot(data=df, x='offered_salary_usd', hue=TARGET, bins=30, kde=True, palette=['#e74c3c', '#2ecc71'])
plt.title('Offered Salary Distribution by Outcome')
plt.xlabel('Offered Salary (USD)')

plt.subplot(1, 2, 2)
sns.histplot(data=df, x='expected_salary_usd', hue=TARGET, bins=30, kde=True, palette=['#e74c3c', '#2ecc71'])
plt.title('Expected Salary Distribution by Outcome')
plt.xlabel('Expected Salary (USD)')

plt.tight_layout()
plt.show()

In [ ]:
# 9. Offer Acceptance Rate by Country & Work Arrangement
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
country_order = df['country'].value_counts().index
sns.barplot(data=df, x='country', y=TARGET, order=country_order, ci=None, palette='Blues_d')
plt.title('Acceptance Rate by Country')
plt.xlabel('Country')
plt.ylabel('Acceptance Rate')
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
sns.barplot(data=df, x='remote_preference', y=TARGET, ci=None, palette='Greens_d')
plt.title('Acceptance Rate by Work Arrangement')
plt.xlabel('Work Arrangement')
plt.ylabel('Acceptance Rate')

plt.tight_layout()
plt.show()

In [ ]:
# 10. Feature Engineering (Salary Gap, Ratio, Hike %, and Experience-to-Age Ratio)
df['salary_offer_gap_usd'] = df['offered_salary_usd'] - df['expected_salary_usd']
df['offer_to_expected_salary_ratio'] = df['offered_salary_usd'] / df['expected_salary_usd'].replace(0, 1)
df['salary_hike_pct'] = (df['salary_offer_gap_usd'] / df['expected_salary_usd'].replace(0, 1)) * 100
df['exp_to_age_ratio'] = df['years_experience'] / df['age'].replace(0, 1)

print('Feature Engineering completed successfully!')
df[['salary_offer_gap_usd', 'offer_to_expected_salary_ratio', 'salary_hike_pct', 'exp_to_age_ratio']].head()

In [ ]:
# 11. Correlation Heatmap
plt.figure(figsize=(10, 8))
numeric_cols = df.select_dtypes(include=[np.number]).columns
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', cbar=True)
plt.title('Correlation Matrix of Numeric Features')
plt.show()

In [ ]:
# 12. Feature Selection: ANOVA F-Test (SelectKBest)
X = df.drop(columns=[TARGET]).copy()
y = df[TARGET]

# Label encode categorical columns temporarily for ANOVA test
for col in X.select_dtypes(include=['object']).columns:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

selector = SelectKBest(score_func=f_classif, k='all')
selector.fit(X, y)

feature_ranking = pd.DataFrame({
    'Feature': X.columns,
    'ANOVA_F_Score': selector.scores_
}).sort_values('ANOVA_F_Score', ascending=False).reset_index(drop=True)

plt.figure(figsize=(10, 5))
sns.barplot(data=feature_ranking, x='ANOVA_F_Score', y='Feature', palette='viridis')
plt.title('Feature Importance Ranking (ANOVA F-Score)')
plt.xlabel('ANOVA F-Score')
plt.show()

feature_ranking

In [ ]:
# 13. Tree-Based Feature Importance (Random Forest)
# Use a fast representative sample (20,000 rows) for tree evaluation
sample_idx = np.random.RandomState(42).choice(len(X), size=20000, replace=False)
rf = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X.iloc[sample_idx], y.iloc[sample_idx])

rf_importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
rf_importances.plot(kind='bar', color='#3498db')
plt.title('Random Forest Feature Importances (Gini Importance)')
plt.ylabel('Importance')
plt.xticks(rotation=45, ha='right')
plt.show()

print('Top Features by Random Forest Importance:')
print(rf_importances)